<a href="https://colab.research.google.com/github/aryan-kumar-shrivastav/ml_workspace_clone/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aryan-kumar-shrivastav/ml_workspace_clone/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Scoring / Ranking

Instead of a basic yes/no classification (which just flags whether a page is declining
or not), this is a ranking problem. An editor has limited time and can't treat every
underperforming page with the same priority. They need an ordered queue so they can
work top-down on the pages where they can make the biggest impact first.

The model will output a priority score for each page to create this ranked list. We'll
evaluate its success based on Precision@K — checking what fraction of the top K pages
flagged are actually worth an editor's time.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target column:** `is_declining_label` (1 when `trend_direction == "down"`, else 0)

**Observed or proxy?** This is a proxy, not a pure observed outcome. The underlying
signal — the real percent change in impressions between the last 30 days and the prior
30 days (`trend_pct`) — is measured. But `is_declining_label` is defined by thresholding
that signal at a >20% drop, a rule choice rather than something intrinsic to the data.
A page down 19% is labeled "not declining"; one down 21% is "declining" — the boundary
is somewhat arbitrary even though the underlying trend is real.

**Leakage constraint:** because the label is built directly from `trend_direction` /
`trend_pct`, those columns can never be used as features — that would leak the answer
into the input. Only pre-decline-window signals (content age, staleness, historical
CTR/position, content type, word count) are valid features.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric:** Precision@K, evaluated at K = 20 and K = 50.

**Why this metric:** the editor works through a queue top-down, not the whole dataset —
so what matters is whether the pages at the *top* of the ranking are genuinely worth
their time, not overall accuracy across all 30,000 pages. Precision@K directly answers
"of the top K pages I'd hand the editor, how many are actually declining?" — which maps
onto their real workflow.

**What "good" looks like:** in Notebook 02, the hand-written rule (stale × visible)
already scored Precision@20 = 0.900 and Precision@50 = 0.680. That sets the bar: a
ranking model only earns its place if it can match or beat those numbers, ideally while
also handling borderline cases (pages that are moderately stale but very visible, or
vice versa) better than the simple rule can.

**Caveat:** these numbers were measured in-sample in Notebook 02, not on a proper
train/test split with client-holdout — so they're a rough baseline, not a guaranteed
target.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis:** one row = one page (`content_id`), aggregated over a trailing
90-day window. Below, a real slice of that dataframe showing the columns relevant to
this lane.

In [13]:
import os
print("Current directory:", os.getcwd())
print(os.listdir("."))

Current directory: /content/ml_workspace_clone
['LICENSE', 'notebooks', 'skills', '.github', 'CLAUDE.md', 'data', '.git', 'submission', 'AGENTS.md', 'docs', 'requirements.txt', 'outputs', 'README.md', 'work', 'DATA_USE.md', 'GUIDE.md', '.gitignore', 'SETUP.md', 'scripts']


In [14]:
import subprocess

REPO_URL = "https://github.com/aryan-kumar-shrivastav/ml_workspace_clone"
REPO_DIR = "ml_workspace_clone"

subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir(REPO_DIR)
print("Now in:", os.getcwd())
print(os.listdir("data"))

Now in: /content/ml_workspace_clone/ml_workspace_clone
['raw']


In [15]:
print(os.listdir("data/raw"))


['content_refresh_anonymized.csv']


In [16]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

lane_cols = [
    "content_id", "client_id", "content_type",
    "days_since_last_update", "impressions_90d", "avg_position",
    "ctr", "word_count", "content_age_days",
    "trend_direction", "is_declining_label"
]

if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

df[lane_cols].head(10)

,content_id,client_id,content_type,days_since_last_update,impressions_90d,avg_position,ctr,word_count,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,20,3803,10.6,0.76,3221.0,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,25,15320,20.3,0.05,2481.0,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,20,12581,36.5,0.09,3515.0,141,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,22,11751,6.2,0.49,NaN,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,14,19140,44.0,0.13,2803.0,263,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,20,3970,8.5,0.03,3080.0,147,down,1
6,content_9a34b442b552,client_8722616204,keyword article,20,20,7.0,0.00,3059.0,90,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,22,1724,21.2,0.06,NaN,445,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,20,32574,46.0,0.09,3807.0,90,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,104,1240,4.9,0.16,NaN,257,down,1


Each row above is **one page** (`content_id`), aggregated over its trailing 90-day
window. This confirms the unit of analysis matches what Section 1 assumed — the model
scores individual pages, and the editor's queue is a ranked list of these same rows.


## 5. Why ML beats a fixed rule here

**Short answer: it's not a clean win — and that's the honest, useful finding.**

In Notebook 02, a one-line hand-written rule (`stale × visible`, using
`days_since_last_update` and `impressions_90d`) already scored Precision@20 = 0.900 and
Precision@50 = 0.680. A depth-2 decision tree, trained on a few more signals, only
managed Precision@50 = 0.550–0.600 in the same test. So for the very top of the queue,
the simple rule is currently *winning*, not losing, to a basic learned model.

**Where a rule genuinely starts to break down, though:**
- The rule only uses two signals (staleness × visibility). It has no way to account for
  `content_type` — and Notebook 01 showed CTR at the same position tier varies by roughly
  6x between content types (`comparison article` vs `feedly article`). A page could be
  stale and visible but already performing fine for its content type, or the reverse.
- The rule treats "stale" and "visible" as independent, equally-weighted flags. Real pages
  don't fall cleanly into four buckets — a page that's moderately stale but highly visible
  may deserve a different priority than one that's very stale but barely visible, and a
  fixed threshold can't express that trade-off; a learned scoring function (which weighs
  and combines features continuously) can.
- As more signals get added (content type, word count, keyword competition, main intent),
  the number of if/else branches a hand-written rule would need to stay accurate grows
  quickly, and nobody would actually maintain a 10-branch nested if-statement by hand.

**Honest conclusion:** a plain rule already captures the easy, top-of-the-list cases well.
ML's potential value here isn't beating the rule at the very top — it's handling the
messier middle of the queue, where staleness, visibility, and content type trade off
against each other in ways a two-variable rule can't represent. This is a hypothesis to
test with more features and a proper held-out split, not a proven result yet.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.